In [6]:
using DynamicPolynomials, JuMP, SumOfSquares, Random, Distributions, Combinatorics, MosekTools

In [2]:
# Function to create a random psd polynomial
function create_psd_b(x, y, p, q; seed::Union{Nothing, Integer}=nothing)
    n = length(x)
    m = length(y)

    # Set the random seed for reproducibility
    if seed !== nothing
        Random.seed!(seed)
        println("Random number generator reseeded with seed: $seed")
    else 
        println("No seed provided, using default random number generator.")
    end

    # Create a random psd polynomial b(x, y)
    M_y = sum([rand(Uniform(-1, 1), n, n) * y[i]^q for i in 1:m])
    println("M_y = ", M_y)
    b = only(x'.^p * M_y' * M_y * x.^p) # b(x, y) is psd
    println("b(x, y) = ", b)
    
    return b
end

#  Function to create a high-degree convex polynomial f(x, y), based on Ahamadi et al. (2013)
function highDegreeConvexPolynomial(x::Vector, y::Vector, p::Integer, q::Integer; seed::Union{Nothing, Integer}=nothing)
    # Get the dimensions of x and y
    n = length(x)
    m = length(y)
    
    # Check the input
    if n < 1 || m < 1
        throw(ArgumentError("n and m must be greater than or equal to 1."))
    end

    if p < 1 || q < 1
        throw(ArgumentError("p and q must be greater than or equal to 1."))
    end

    if p % 2 == 0 || q % 2 == 0
        throw(ArgumentError("p and q must be odd integers."))
    end 

    # Create a psd polynomial b(x, y)
    b = create_psd_b(x, y, p, q; seed=seed)

    # Get the values of λ, μ, and ν
    lambda = maximum([maximum.(coefficients.(differentiate(differentiate(b, x), y))); - minimum.(coefficients.(differentiate(differentiate(b, x), y)))])
    mu = p == 1 ? 0 : maximum([maximum.(coefficients.(differentiate(b, x, 2))); - minimum.(coefficients.(differentiate(b, x, 2)))])
    nu = q == 1 ? 0 : maximum([maximum.(coefficients.(differentiate(b, y, 2))); - minimum.(coefficients.(differentiate(b, y, 2)))])

    println("lambda = ", lambda)
    println("mu = ", mu)
    println("nu = ", nu)

    # Construct g(x, y), h(x, y), and w(x, y)
    g = m^2 * lambda / (2 * p * (2*p - 1)) * sum(prod.(collect(with_replacement_combinations(x.^(2*p), 2)))) + n^2 * lambda / (2 * q * (2*q - 1)) * sum(prod.(collect(with_replacement_combinations(y.^(2*q), 2))))
    if p == 1 && q == 1
        h = 0
        w = 0
    elseif p > 1 && q == 1
        h = m * (m+1) * mu / (4 * p * (p+1)) * sum(prod.(collect(with_replacement_combinations(x.^(p+1), 2)))) + mu / (2 * (p-1) * (p-2)) * sum(prod.(collect(with_replacement_combinations(x.^(p-1), 2)))) * sum(prod.(collect(with_replacement_combinations(y.^(2*q), 2))))
        w = 2 * m^2 * q * mu / ((p-1) * (p-2) * (2*p-3)) * sum(prod.(collect(with_replacement_combinations(x.^(2*p-2), 2)))) + n^2 * mu / ((p-2) * (4*q-1)) * sum(prod.(collect(with_replacement_combinations(y.^(4*q), 2))))
    elseif p == 1 && q > 1
        h = n * (n+1) * nu / (4 * q * (q+1)) * sum(prod.(collect(with_replacement_combinations(y.^(q+1), 2)))) + nu / (2 * (q-1) * (q-2)) * sum(prod.(collect(with_replacement_combinations(y.^(q-1), 2)))) * sum(prod.(collect(with_replacement_combinations(x.^(2*p), 2))))
        w = 2 * n^2 * p * nu / ((q-1) * (q-2) * (2*q-3)) * sum(prod.(collect(with_replacement_combinations(y.^(2*q-2), 2)))) + m^2 * nu / ((q-2) * (4*p-1)) * sum(prod.(collect(with_replacement_combinations(x.^(4*p), 2))))
    else    # p > 1 && q > 1
        h = m * (m+1) * mu / (4 * p * (p+1)) * sum(prod.(collect(with_replacement_combinations(x.^(p+1), 2)))) + mu / (2 * (p-1) * (p-2)) * sum(prod.(collect(with_replacement_combinations(x.^(p-1), 2)))) * sum(prod.(collect(with_replacement_combinations(y.^(2*q), 2)))) + n * (n+1) * nu / (4 * q * (q+1)) * sum(prod.(collect(with_replacement_combinations(y.^(q+1), 2)))) + nu / (2 * (q-1) * (q-2)) * sum(prod.(collect(with_replacement_combinations(y.^(q-1), 2)))) * sum(prod.(collect(with_replacement_combinations(x.^(2*p), 2))))
        w = 2 * m^2 * q * mu / ((p-1) * (p-2) * (2*p-3)) * sum(prod.(collect(with_replacement_combinations(x.^(2*p-2), 2)))) + m^2 * nu / ((q-2) * (4*p-1)) * sum(prod.(collect(with_replacement_combinations(x.^(4*p), 2)))) + 2 * n^2 * p * nu / ((q-1) * (q-2) * (2*q-3)) * sum(prod.(collect(with_replacement_combinations(y.^(2*q-2), 2)))) + n^2 * mu / ((p-2) * (4*q-1)) * sum(prod.(collect(with_replacement_combinations(y.^(4*q), 2))))
    end 

    println("g(x, y) = ", g)
    println("h(x, y) = ", h)
    println("w(x, y) = ", w)
    
    # Construct f(x, y)
    f = b + g + h + w
    println("f(x, y) = ", f)

    return f
end

highDegreeConvexPolynomial (generic function with 1 method)

Example single player, SOS-Monotone game $\mathcal{G}_{\rm SOS}$, with degree 4. 

In [3]:
@polyvar x[1:2] y[1:2]
p = 1
q = 1

f = highDegreeConvexPolynomial(x, y, p, q; seed=1234)

Random number generator reseeded with seed: 1234
M_y = Polynomial{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, Graded{LexOrder}, Float64}[-0.2937767112015759y₂ - 0.3480465422728103y₁ 0.9062492545696843y₂ - 0.5628266903623387y₁; -0.21148926516829847y₂ + 0.0981022726311338y₁ 0.5910938950694389y₂ + 0.7884908564019766y₁]
b(x, y) = 1.1706797041964692*x[2]^2*y[2]^2 - 0.0879782741110201*x[2]^2*y[1]*y[2] + 0.9384917140137463*x[2]^2*y[1]^2 - 0.782489878100129*x[1]*x[2]*y[2]^2 - 0.5176824855737008*x[1]*x[2]*y[1]*y[2] + 0.546485256882729*x[1]*x[2]*y[1]^2 + 0.13103246532584098*x[1]^2*y[2]^2 + 0.1630007819677758*x[1]^2*y[1]*y[2] + 0.13076045148345242*x[1]^2*y[1]^2
lambda = 4.682718816785877
mu = 0
nu = 0
g(x, y) = 9.365437633571753*y[2]^4 + 9.365437633571753*y[1]^2*y[2]^2 + 9.365437633571753*y[1]^4 + 9.365437633571753*x[2]^4 + 9.365437633571753*x[1]^2*x[2]^2 + 9.365437633571753*x[1]^4
h(x, y) = 0
w(x, y) = 0
f(x, y) = 9.365437633571753*y[2]^4 + 9.365437633571753*y[1]^2*y[2]^2 +

9.365437633571753y₂⁴ + 9.365437633571753y₁²y₂² + 9.365437633571753y₁⁴ + 1.1706797041964692x₂²y₂² - 0.0879782741110201x₂²y₁y₂ + 0.9384917140137463x₂²y₁² + 9.365437633571753x₂⁴ - 0.782489878100129x₁x₂y₂² - 0.5176824855737008x₁x₂y₁y₂ + 0.546485256882729x₁x₂y₁² + 0.13103246532584098x₁²y₂² + 0.1630007819677758x₁²y₁y₂ + 0.13076045148345242x₁²y₁² + 9.365437633571753x₁²x₂² + 9.365437633571753x₁⁴

In [4]:
gs = [x; 1 - sum(x); 1 - sum(x.^2); y; 1 - sum(y); 1 - sum(y.^2)]
hs = [sum(x) - 1; sum(y) - 1]
Sg = basic_semialgebraic_set(FullSpace(), gs) 
Sh = algebraic_set(hs)
Sx = intersect(Sh, Sg)

Basic semialgebraic Set defined by 2 equalities
 -1//1 + x[2] + x[1] = 0
 -1//1 + y[2] + y[1] = 0
8 inequalities
 x[1] ≥ 0
 x[2] ≥ 0
 1//1 - x[2] - x[1] ≥ 0
 1//1 - x[2]^2 - x[1]^2 ≥ 0
 y[1] ≥ 0
 y[2] ≥ 0
 1//1 - y[2] - y[1] ≥ 0
 1//1 - y[2]^2 - y[1]^2 ≥ 0


The problem, though high dimensional, is solved with a single SDP

In [7]:
model = SOSModel(Mosek.Optimizer)
@variable(model, t)
@objective(model, Min, t)
@constraint(model, c, f <= t, domain = Sx, maxdegree = 4)
optimize!(model)
println("Solution: $(value(t))")

v = moment_matrix(model[:c])
nu = atomic_measure(v, 0.5e-1)
print(nu)

Problem
  Name                   :                 
  Objective sense        : minimize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 15              
  Affine conic cons.     : 0               
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 1               
  Matrix variables       : 9 (scalarized: 240)
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.00            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0        

Degree 12 example

In [18]:
@polyvar x[1:2] y[1:2]
p = 1
q = 3

f = highDegreeConvexPolynomial(x, y, p, q; seed=1)

Random number generator reseeded with seed: 1
M_y = Polynomial{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, Graded{LexOrder}, Float64}[0.8298580073256627y₂³ - 0.8532672910614143y₁³ 0.5403606957713327y₂³ + 0.397653367382937y₁³; -0.6143837675082491y₂³ - 0.3015170208856277y₁³ 0.5610385273503726y₂³ + 0.25652948068500336y₁³]
b(x, y) = 0.6067539107059536*x[2]^2*y[2]^6 + 0.7175983446806546*x[2]^2*y[1]^3*y[2]^3 + 0.22393557505150657*x[2]^2*y[1]^6 + 0.2074593721582101*x[1]*x[2]*y[2]^6 - 0.9156929804061068*x[1]*x[2]*y[1]^3*y[2]^3 - 0.8333052327075348*x[1]*x[2]*y[1]^6 + 1.06613172610015*x[1]^2*y[2]^6 - 1.0456870612336324*x[1]^2*y[1]^3*y[2]^3 + 0.8189775838790283*x[1]^2*y[1]^6
lambda = 12.793580713201798
mu = 0
nu = 31.983951783004496
g(x, y) = 25.587161426403597*x[2]^4 + 25.587161426403597*x[1]^2*x[2]^2 + 25.587161426403597*x[1]^4 + 1.7058107617602398*y[2]^12 + 1.7058107617602398*y[1]^6*y[2]^6 + 1.7058107617602398*y[1]^12
h(x, y) = 3.997993972875562*y[2]^8 + 3.997993972875562

25.587161426403597x₂⁴ + 25.587161426403597x₁²x₂² + 25.587161426403597x₁⁴ + 46.643263016881555y₂⁸ + 46.643263016881555y₁⁴y₂⁴ + 46.643263016881555y₁⁸ + 0.6067539107059536x₂²y₂⁶ + 0.7175983446806546x₂²y₁³y₂³ + 0.22393557505150657x₂²y₁⁶ + 7.995987945751124x₂⁴y₂⁴ + 7.995987945751124x₂⁴y₁²y₂² + 7.995987945751124x₂⁴y₁⁴ + 42.64526904400599x₂⁸ + 0.2074593721582101x₁x₂y₂⁶ - 0.9156929804061068x₁x₂y₁³y₂³ - 0.8333052327075348x₁x₂y₁⁶ + 1.06613172610015x₁²y₂⁶ - 1.0456870612336324x₁²y₁³y₂³ + 0.8189775838790283x₁²y₁⁶ + 7.995987945751124x₁²x₂²y₂⁴ + 7.995987945751124x₁²x₂²y₁²y₂² + 7.995987945751124x₁²x₂²y₁⁴ + 7.995987945751124x₁⁴y₂⁴ + 7.995987945751124x₁⁴y₁²y₂² + 7.995987945751124x₁⁴y₁⁴ + 42.64526904400599x₁⁴x₂⁴ + 42.64526904400599x₁⁸ + 1.7058107617602398y₂¹² + 1.7058107617602398y₁⁶y₂⁶ + 1.7058107617602398y₁¹²

In [20]:
gs = [x; 1 - sum(x); 1 - sum(x.^2); y; 1 - sum(y); 1 - sum(y.^2)]
hs = [sum(x) - 1; sum(y) - 1]
Sg = basic_semialgebraic_set(FullSpace(), gs) 
Sh = algebraic_set(hs)
Sx = intersect(Sh, Sg)

model = SOSModel(Mosek.Optimizer)
@variable(model, t)
@objective(model, Min, t)
@constraint(model, c, f <= t, domain = Sx, maxdegree = 12)
optimize!(model)
println("Solution: $(value(t))")

v = moment_matrix(model[:c])
nu = atomic_measure(v, 0.5e-1)
print(nu)

Problem
  Name                   :                 
  Objective sense        : minimize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 91              
  Affine conic cons.     : 0               
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 1               
  Matrix variables       : 9 (scalarized: 86163)
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.00            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0      